# BÀI THỰC HÀNH 2 - BÀI 2 (TRANG 37, 38)
## KHAI THÁC LUẬT KẾT HỢP VỚI TẬP DỮ LIỆU ONLINE RETAIL
- **Học viên:** Nguyễn Minh Vương
- **MSSV:** 6451071090
- **Môn học:** Khai thác dữ liệu

In [1]:
# Cài đặt thư viện nếu chạy trên Google Colab
# !pip install pandas numpy matplotlib seaborn mlxtend openpyxl gdown

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
%matplotlib inline

# Tự động tải thư mục dataset từ link Google Drive
if not os.path.exists("dataset") or len(os.listdir("dataset")) == 0:
    try:
        import gdown
        drive_folder_url = "https://drive.google.com/drive/folders/1rv5SiO5zFlU-cyEXul09OpCvTXrdjG1m"
        print("Đang tải dataset từ Google Drive...")
        gdown.download_folder(drive_folder_url, output="dataset", quiet=False, use_cookies=False)
        print("✅ Tải dataset thành công!")
    except Exception as e:
        print("Lưu ý: Nếu chưa có thư viện gdown, hãy chạy lệnh: !pip install gdown")

print("Nguyễn Minh Vương - 6451071090")


Nguyễn Minh Vương - 6451071090


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Câu 1: Đọc dữ liệu vào chương trình

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

df = pd.read_excel('dataset/Online Retail.xlsx')

print(f"Kích thước tập dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột\n")

print("--- 5 dòng đầu tiên ---")
display(df.head())

print("\n--- Cấu trúc dữ liệu (info) ---")
df.info()


### Câu 2: Tiền xử lý dữ liệu: cắt khoảng trắng cột Description, xóa dòng thiếu InvoiceNo và đổi sang kiểu chuỗi

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

df['Description'] = df['Description'].str.strip()
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')

print("Kiểm tra 5 dòng đầu sau khi làm sạch Description và InvoiceNo:")
display(df.head())


### Câu 3: Xóa các hóa đơn tín dụng (hóa đơn chứa ký tự 'C' trong mã InvoiceNo)

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

so_dong_truoc = len(df)
df = df[~df['InvoiceNo'].str.contains('C')]
so_dong_sau = len(df)

print(f"Số dòng trước khi xóa hóa đơn tín dụng: {so_dong_truoc}")
print(f"Số dòng sau khi xóa: {so_dong_sau}")
print(f"Đã xóa {so_dong_truoc - so_dong_sau} dòng hóa đơn tín dụng.")


### Câu 4: Thống kê số dòng dữ liệu theo từng quốc gia

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

country_counts = df['Country'].value_counts()
print("Top 10 quốc gia có số lượng giao dịch nhiều nhất:")
print(country_counts.head(10))

plt.figure(figsize=(10, 8))
df['Country'].value_counts().plot(kind='barh', color='cornflowerblue')
plt.title('Thống kê số dòng dữ liệu theo từng quốc gia')
plt.xlabel('Số lượng giao dịch (dòng)')
plt.ylabel('Quốc gia')
plt.gca().invert_yaxis()
plt.show()


### Câu 5: Lấy ra dữ liệu hóa đơn từ nước Anh 'United Kingdom' và gom nhóm Quantity theo InvoiceNo và Description

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

grouped_uk = df[df['Country'] == "United Kingdom"].groupby(['InvoiceNo', 'Description'])['Quantity']

print("Kết quả sau khi gom nhóm dữ liệu (10 phần tử đầu):")
display(grouped_uk.sum().head(10))


### Câu 6: Chuyển đổi dữ liệu về dạng hot encoding, với mỗi dòng dữ liệu là một hóa đơn

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

basket = (df[df['Country'] == "United Kingdom"]
          .groupby(['InvoiceNo', 'Description'])['Quantity'].sum()
          .unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

print(f"Kích thước ma trận giỏ hàng (basket): {basket.shape[0]} hóa đơn, {basket.shape[1]} mặt hàng")
print("Hiển thị 5 dòng và 8 cột đầu tiên của ma trận giỏ hàng:")
display(basket.iloc[:5, :8])


### Câu 7: Chuyển đổi dữ liệu từ dạng hot encoding thành one-hot encoding (0 hoặc 1)

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

def encode_data(datapoint):
    if datapoint <= 0:
        return 0
    if datapoint >= 1:
        return 1

basket = basket.map(encode_data) if hasattr(basket, 'map') else basket.applymap(encode_data)

print("Ma trận dữ liệu One-hot encoding sau khi chuẩn hóa thành 0 và 1:")
display(basket.iloc[:5, :8])


### Câu 8: Xóa cột 'POSTAGE' khỏi tập dữ liệu

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

if 'POSTAGE' in basket.columns:
    basket.drop('POSTAGE', inplace=True, axis=1)
    print("Đã xóa thành công cột 'POSTAGE' khỏi ma trận giỏ hàng.")
else:
    print("Cột 'POSTAGE' không tồn tại trong tập dữ liệu giỏ hàng.")


### Câu 9: Tìm tập phổ biến bằng thuật toán Apriori với min_sup = 3%

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

itemsets_apriori = apriori(basket, min_support=0.03, use_colnames=True)

print(f"Tổng số tập mục phổ biến tìm được bằng Apriori: {len(itemsets_apriori)}")
print("\n10 phần tử đầu tiên trong tập phổ biến:")
display(itemsets_apriori.head(10))


### Câu 10: Tạo luật kết hợp với min_conf = 50% và in ra các luật này

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

rules = association_rules(itemsets_apriori, metric="confidence", min_threshold=0.5)

print(f"Tổng số luật kết hợp tìm được: {len(rules)}\n")

rules_print = rules.copy()
rules_print['antecedents'] = rules_print['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_print['consequents'] = rules_print['consequents'].apply(lambda x: ', '.join(list(x)))

for i in range(len(rules_print)):
    print(f"Luật {i+1}: {rules_print.loc[i, 'antecedents']}  ==>  {rules_print.loc[i, 'consequents']}")
    print(f"         Support: {rules_print.loc[i, 'support']:.4f} | Confidence: {rules_print.loc[i, 'confidence']:.4f} | Lift: {rules_print.loc[i, 'lift']:.4f}\n")


### Câu 11: Biểu diễn độ tin cậy và độ hỗ trợ của tập luật lên đồ thị phân tán (scatter plot)

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

plt.figure(figsize=(8, 6))
plt.title('Association Rules (Support vs Confidence)', fontsize=14)
plt.xlabel('Support', fontsize=12)
plt.ylabel('Confidence', fontsize=12)

sns.regplot(x=rules['support'], y=rules['confidence'], fit_reg=False, color='darkorchid', scatter_kws={'s': 60})
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### Câu 12: Tìm tập phổ biến và luật kết hợp bằng thuật toán FP-Growth với min_sup = 3%, min_conf = 50% và so sánh với Apriori

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

# Tìm tập phổ biến bằng thuật toán FP-Growth
itemsets_fpgrowth = fpgrowth(basket, min_support=0.03, use_colnames=True)
print(f"Số lượng tập phổ biến bằng FP-Growth: {len(itemsets_fpgrowth)}")
print(f"Số lượng tập phổ biến bằng Apriori:   {len(itemsets_apriori)}")

# Sinh luật kết hợp từ kết quả FP-Growth
rules_fpgrowth = association_rules(itemsets_fpgrowth, metric="confidence", min_threshold=0.5)
print(f"Số lượng luật kết hợp từ FP-Growth:    {len(rules_fpgrowth)}")
print(f"Số lượng luật kết hợp từ Apriori:      {len(rules)}")

print("\n--- NHẬN XÉT SO SÁNH HAI THUẬT TOÁN ---")
print("1. Số lượng tập phổ biến và tập luật kết hợp tìm được từ cả hai thuật toán là hoàn toàn trùng khớp.")
print("2. Về thuật toán:")
print("   - Apriori phải sinh các tập ứng viên (candidate generation) và quét qua cơ sở dữ liệu nhiều lần.")
print("   - FP-Growth nén cơ sở dữ liệu thành cấu trúc cây FP-tree và khai thác đệ quy, chỉ quét CSDL 2 lần, do đó đạt hiệu năng cao hơn rất nhiều.")
